# 19｜大型語言模型_生成與聊天系統

在固定 logits 上比較 greedy 與 top-k 抽樣，觀察溫度的影響。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/19_大型語言模型_生成與聊天系統.md`，目前 30 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

條件生成/top-p、答案遮罩/SFT/DPO、本地檢索與工具介面。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 19-03 | 7、8 | 課堂小實驗 |
| 19-04 | 3、8 | 執行結果圖 |
| 19-05 | 4、5、6、7、8、9、10、11、12、13、30 | 1. 前綴條件、停止條件與 top-p |
| 19-06 | 14、15、16、17、18、19、20、30 | 2. SFT 位移、答案遮罩與 DPO |
| 19-07 | 21、22、23、24、25、26、27、28、29、30 | 3. 可檢查的檢索、引用與工具介面 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 19-01｜投影片 19：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 19-02｜投影片 19：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

rng = np.random.default_rng(SEED)

## 課堂小實驗

固定 logits 的多次抽樣只是分布示範，各次互相獨立；它不是會根據前綴更新機率的語言模型生成。下方用小轉移表示範真正的自迴歸迴圈。

In [ ]:
# 程式 19-03｜投影片 19：7、8 頁
vocab = np.array(['資料', '模型', '學習', '風險', '答案'])
logits = np.array([1.2, 2.4, 1.8, 0.4, 1.0])
def probs(temp):
    z = logits / temp; p = np.exp(z-z.max()); return p/p.sum()
def sample_top_k(temp=1.0, k=3, n=8):
    p = probs(temp); ids = np.argsort(p)[-k:]; q = p[ids]/p[ids].sum()
    return rng.choice(vocab[ids], size=n, p=q)
rows = pd.DataFrame({'token': vocab, 'p_T0.7': probs(.7), 'p_T1.3': probs(1.3)})
print('greedy:', vocab[np.argmax(logits)])
print('top-k sample:', ' '.join(sample_top_k()))
rows.round(3)

## 執行結果圖

In [ ]:
# 程式 19-04｜投影片 19：3、8 頁
x = np.arange(len(vocab)); w = .36
plt.bar(x-w/2, probs(.7), w, label='temperature 0.7')
plt.bar(x+w/2, probs(1.3), w, label='temperature 1.3')
plt.xticks(x, ['data','model','learning','risk','answer']); plt.ylabel('probability'); plt.title('Temperature changes distribution sharpness')
plt.legend()
savefig('19_sampling_demo')
report('llm_sampling', {'vocab': vocab.tolist(), 'p_07': probs(.7).tolist(), 'p_13': probs(1.3).tolist()})

## 1. 前綴條件、停止條件與 top-p

用小詞表條件轉移機率取代昂貴模型，逐步更新前綴。top-p 保留累積機率首次達標的 token，重新正規化。聊天模板是示意格式，實際模型須使用自己的模板。

In [ ]:
# 程式 19-05｜投影片 19：4、5、6、7、8、9、10、11、12、13、30 頁
labels19=['<bos>','learn','with','care','<eos>']
transition19=np.array([[0,.8,.1,.1,0],[0,.05,.75,.1,.1],[0,.05,.05,.8,.1],[0,.05,.05,.1,.8],[0,0,0,0,1.]])
def nucleus19(p,cut=.9):
    assert 0<cut<=1
    ids=np.argsort(p)[::-1];count=np.searchsorted(np.cumsum(p[ids]),cut)+1;chosen=ids[:count]
    out=np.zeros_like(p);out[chosen]=p[chosen];return out/out.sum()
state19=0;generated19=[];logprob19=0.
for step19 in range(8):
    base19=transition19[state19];distribution19=nucleus19(base19,.9)
    token19=int(rng.choice(len(labels19),p=distribution19));logprob19+=np.log(base19[token19]);generated19.append(token19);state19=token19
    if token19==4:break
print('Autoregressive tokens:',[labels19[i] for i in generated19],'base-model sequence log probability:',logprob19)
assert len(generated19)<=8
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].imshow(transition19,vmin=0,vmax=1,cmap='Blues');axes[0].set(xlabel='next token',ylabel='last token',title='Conditional distribution')
axes[1].bar(np.arange(5)-.15,probs(1),.3,label='full');axes[1].bar(np.arange(5)+.15,nucleus19(probs(1),.8),.3,label='top-p=.8');axes[1].legend();savefig('c19_autoregressive_sampling')
history19=[('system','回答只使用所附課程規則。'),('user','作業何時繳交？')]
serialized19='\n'.join(f'<{role}>{content}</{role}>' for role,content in history19)+'\n<assistant>'
print('Illustrative chat serialization:',serialized19)
parameters19=7_000_000_000;print('7B weights-only GiB FP32/BF16/int8:',[parameters19*b/2**30 for b in [4,2,1]])

## 2. SFT 位移、答案遮罩與 DPO

只加總回答位置的 log probability；padding 與 prompt 不進回答損失。DPO 比較偏好／拒絕回答相對固定參考模型的 log probability 差，不直接學一個外部 reward model。

In [ ]:
# 程式 19-06｜投影片 19：14、15、16、17、18、19、20、30 頁
def logsoftmax19(z):
    z=z-z.max(-1,keepdims=True);return z-np.log(np.exp(z).sum(-1,keepdims=True))
rng19=np.random.default_rng(SEED);input_ids19=np.array([[0,1,2,3,4,0]])
answer_mask19=np.array([[0,0,0,1,1,0]],bool)
logits19=rng19.normal(size=(1,6,5));lp19=logsoftmax19(logits19[:,:-1])
next19=input_ids19[:,1:];token_lp19=np.take_along_axis(lp19,next19[:,:,None],axis=2)[:,:,0]
selected19=answer_mask19[:,1:];sum_answer19=(token_lp19*selected19).sum(1);sft_loss19=-sum_answer19.sum()/selected19.sum()
assert selected19.sum()==2
print('Answer token log probabilities:',token_lp19[selected19],'sum:',sum_answer19,'SFT mean:',sft_loss19)
beta19=1.;delta_c19=.4;delta_r19=-.2
margin19=beta19*(delta_c19-delta_r19);dpo19=np.logaddexp(0,-margin19)
assert np.isclose(dpo19,.43748795) and np.isclose(np.logaddexp(0,0),np.log(2))
margin_grid19=np.linspace(-4,4,200);plt.plot(margin_grid19,np.logaddexp(0,-margin_grid19));plt.scatter([margin19],[dpo19]);plt.xlabel('beta * relative preference margin');plt.ylabel('DPO loss');savefig('c19_dpo_loss')
# 只改玩具偏好 margin，固定 reference；非 LLM 訓練
m19=0.;before19=np.logaddexp(0,-m19)
for _ in range(20):m19+=.1/(1+np.exp(m19))
assert np.logaddexp(0,-m19)<before19
print('DPO worked example / optimized toy margin:',dpo19,m19)

## 3. 可檢查的檢索、引用與工具介面

本格只用內嵌虛構課規與本地純函式。檢索器選證據，答案回傳原文及來源 ID；工具輸入須通過欄位和型態檢查。這不是外部服務、完整 MCP 或生產用聊天模型。

In [ ]:
# 程式 19-07｜投影片 19：21、22、23、24、25、26、27、28、29、30 頁
docs19=[{'id':'rule-1','text':'作業於每週五晚上繳交。'},{'id':'rule-2','text':'報告須包含資料來源與評估指標。'},{'id':'rule-3','text':'補交作業須註明原因。'}]
query19='作業何時繳交'
# 用字元 bigram 重疊的小型檢索；正式系統另需檢索評估
bigrams19=lambda s:set(zip(s,s[1:]))
q19=bigrams19(query19);retrieved19=sorted(docs19,key=lambda d:len(q19&bigrams19(d['text'])),reverse=True)[:2]
answer19={'answer':retrieved19[0]['text'],'source_ids':[retrieved19[0]['id']]}
assert set(answer19['source_ids'])<={d['id'] for d in retrieved19}
print('Retrieved evidence / extractive answer:',retrieved19,answer19)
def execute_tool19(call):
    assert set(call)=={'name','arguments'} and call['name']=='add'
    args=call['arguments'];assert set(args)=={'a','b'}
    assert all(type(args[k]) in (int,float) and np.isfinite(args[k]) for k in args)
    return {'result':args['a']+args['b']}
proposal19={'name':'add','arguments':{'a':2,'b':3}}
print('Validated local tool result:',execute_tool19(proposal19))
assert execute_tool19(proposal19)=={'result':5}
# 同一結構可接 task prefix；此處僅整理 text-to-text 輸入，不生成摘要
print('Text-to-text task input:','summarize: '+retrieved19[0]['text'])

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。